# Convert FunctionGemma to Flutter Gemma `.task`

This notebook converts the fine-tuned banking FunctionGemma model from Hugging Face SafeTensors into a MediaPipe Task Bundle (`.task`) that can be installed with `flutter_gemma` on Android/iOS.

Default source and target repo: `phattrandeveloper/functiongemma-270m-function-calling`.

## Verified guide path

I based this on Google's current Gemma conversion guide:

1. Download or point at an existing Hugging Face Gemma/FunctionGemma model folder.
2. Use `litert-torch` to convert the PyTorch/SafeTensors checkpoint into a multi-signature LiteRT `.tflite` model.
3. Use `mediapipe.tasks.python.genai.bundler` to bundle the `.tflite` model and `tokenizer.model` into a `.task` file.
4. Upload the `.task` artifact to Hugging Face and use the `/resolve/main/...task` URL in Flutter Gemma.

References:

- Google Gemma conversion guide: https://ai.google.dev/gemma/docs/conversions/hf-to-mediapipe-task
- Google MediaPipe LLM Inference guide: https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference
- Flutter Gemma package docs: https://pub.dev/packages/flutter_gemma

## Runtime notes

Google notes that the LiteRT Torch Generative converter is CPU-only and intended for Linux machines with substantial RAM. Use a Linux VM/Colab/runtime for the conversion step if local macOS conversion fails.

The generated `.task` file is the Flutter Gemma mobile artifact. Do not point Flutter Gemma at `model.safetensors`.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()

SOURCE_MODEL_ID = os.getenv(
    "SOURCE_MODEL_ID", "phattrandeveloper/functiongemma-270m-function-calling"
)
TARGET_REPO_ID = os.getenv("TARGET_REPO_ID", SOURCE_MODEL_ID)
ARTIFACT_NAME = os.getenv("ARTIFACT_NAME", "functiongemma-270m-function-calling")
TOKENIZER_MODEL_ID = os.getenv("TOKENIZER_MODEL_ID", "google/functiongemma-270m-it")

# Supported by litert_torch.generative.examples.gemma3.gemma3.
GEMMA3_SIZE = os.getenv("GEMMA3_SIZE", "270m")  # "270m" or "1b"

# Tune these for your target device. Larger KV cache means larger memory use.
PREFILL_SEQ_LEN = int(os.getenv("PREFILL_SEQ_LEN", "2048"))
KV_CACHE_MAX_LEN = int(os.getenv("KV_CACHE_MAX_LEN", "4096"))
QUANTIZE = os.getenv("LITERT_QUANTIZE", "dynamic_int8")

# Side-effect flags. Set UPLOAD_TO_HF=1 before running the publish cell.
DOWNLOAD_MODEL = os.getenv("DOWNLOAD_MODEL", "1") == "1"
SKIP_CONVERSION_IF_TFLITE_EXISTS = os.getenv("SKIP_CONVERSION_IF_TFLITE_EXISTS", "1") == "1"
UPLOAD_TO_HF = os.getenv("UPLOAD_TO_HF", "0") == "1"
UPLOAD_RAW_TFLITE = os.getenv("UPLOAD_RAW_TFLITE", "1") == "1"
UPLOAD_CARD_TO_ROOT = os.getenv("UPLOAD_CARD_TO_ROOT", "0") == "1"
PRIVATE_REPO = os.getenv("HF_PRIVATE_REPO", "0") == "1"

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

OUTPUT_ROOT = REPO_ROOT / "artifacts" / "flutter_gemma" / ARTIFACT_NAME
HF_MODEL_DIR = OUTPUT_ROOT / "hf_model"
LITERT_DIR = OUTPUT_ROOT / "litert"
TOKENIZER_DIR = OUTPUT_ROOT / "tokenizer"
TASK_FILE = OUTPUT_ROOT / f"{ARTIFACT_NAME}.task"
METADATA_FILE = OUTPUT_ROOT / "conversion_metadata.json"
CARD_FILE = OUTPUT_ROOT / "flutter_gemma_README.md"

for path in [OUTPUT_ROOT, HF_MODEL_DIR, LITERT_DIR, TOKENIZER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Source model:", SOURCE_MODEL_ID)
print("Target Hugging Face repo:", TARGET_REPO_ID)
print("Tokenizer fallback repo:", TOKENIZER_MODEL_ID)
print("Output root:", OUTPUT_ROOT)
print("Task output:", TASK_FILE)
print("Platform:", platform.platform())
if platform.system() != "Linux":
    print("Note: Google's converter guidance is Linux-oriented; use Linux/Colab if conversion fails here.")

## Install/check dependencies

In a fresh Linux or Colab runtime, run this once before continuing:

```python
%pip install -U "litert-torch>=0.8.0" "mediapipe>=0.10.14" "huggingface_hub[cli]>=0.25.0"
```

In [ ]:
required_modules = {
    "huggingface_hub": "huggingface_hub[cli]>=0.25.0",
    "litert_torch": "litert-torch>=0.8.0",
    "mediapipe": "mediapipe>=0.10.14",
}

missing = [
    package_spec
    for module_name, package_spec in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Missing conversion dependencies:")
    for package_spec in missing:
        print("-", package_spec)
    print("\nInstall with:")
    print('%pip install -U "litert-torch>=0.8.0" "mediapipe>=0.10.14" "huggingface_hub[cli]>=0.25.0"')
else:
    print("All conversion dependencies are importable.")

## Download the Hugging Face model

Set `HF_TOKEN` if the model or base Gemma license requires authentication. If the model is already present locally, set `DOWNLOAD_MODEL=0` and place it under `artifacts/flutter_gemma/<artifact-name>/hf_model`.

In [ ]:
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face with token from the environment.")
else:
    print("HF_TOKEN is not set. Public repos may still download; gated/private repos will fail.")

if DOWNLOAD_MODEL:
    snapshot_download(
        repo_id=SOURCE_MODEL_ID,
        local_dir=str(HF_MODEL_DIR),
        token=HF_TOKEN,
    )
else:
    print("DOWNLOAD_MODEL=0; using existing local model directory.")

print("Model directory:", HF_MODEL_DIR)
print("Top-level files:")
for path in sorted(HF_MODEL_DIR.iterdir())[:30]:
    print("-", path.name)

In [ ]:
def find_one(pattern: str, root: Path) -> Path | None:
    matches = sorted(root.rglob(pattern))
    return matches[0] if matches else None


from huggingface_hub import snapshot_download

TOKENIZER_MODEL = find_one("tokenizer.model", HF_MODEL_DIR)
SAFETENSOR_FILES = sorted(HF_MODEL_DIR.rglob("*.safetensors"))

if TOKENIZER_MODEL is None:
    print("No tokenizer.model found in the fine-tuned model directory.")
    print("Trying to fetch a compatible SentencePiece tokenizer from base Gemma repos...")

    candidate_tokenizer_repos = []
    for repo_id in [SOURCE_MODEL_ID, TOKENIZER_MODEL_ID, "google/gemma-3-270m-it"]:
        if repo_id and repo_id not in candidate_tokenizer_repos:
            candidate_tokenizer_repos.append(repo_id)

    tokenizer_errors = []
    for repo_id in candidate_tokenizer_repos:
        local_tokenizer_dir = TOKENIZER_DIR / repo_id.replace("/", "__")
        try:
            snapshot_download(
                repo_id=repo_id,
                local_dir=str(local_tokenizer_dir),
                allow_patterns=["tokenizer.model"],
                token=HF_TOKEN,
            )
            TOKENIZER_MODEL = find_one("tokenizer.model", local_tokenizer_dir)
            if TOKENIZER_MODEL is not None:
                print(f"Using tokenizer.model from {repo_id}: {TOKENIZER_MODEL}")
                break
            tokenizer_errors.append(f"{repo_id}: tokenizer.model was not present")
        except Exception as exc:
            tokenizer_errors.append(f"{repo_id}: {type(exc).__name__}: {exc}")

if TOKENIZER_MODEL is None:
    raise FileNotFoundError(
        "Could not locate tokenizer.model. MediaPipe bundling needs the SentencePiece tokenizer model. "
        "Set HF_TOKEN for gated Google repos, or set TOKENIZER_MODEL_ID to a repo/local-compatible HF repo "
        "that contains tokenizer.model. Attempts:\n- " + "\n- ".join(tokenizer_errors)
    )
if not SAFETENSOR_FILES:
    raise FileNotFoundError(f"Could not find any .safetensors weights under {HF_MODEL_DIR}.")

print("Tokenizer model:", TOKENIZER_MODEL)
print(f"SafeTensors files: {len(SAFETENSOR_FILES)}")
for path in SAFETENSOR_FILES[:10]:
    print("-", path.relative_to(HF_MODEL_DIR))

## Convert SafeTensors to LiteRT `.tflite`

For FunctionGemma 270M, use the Gemma 3 270M builder. The converter writes the LiteRT model into `LITERT_DIR`; the next cell discovers the exact emitted filename.

In [ ]:
existing_tflites = sorted(LITERT_DIR.rglob("*.tflite"))

if existing_tflites and SKIP_CONVERSION_IF_TFLITE_EXISTS:
    print("Found existing LiteRT file(s); skipping conversion.")
    for path in existing_tflites:
        print("-", path)
else:
    from litert_torch.generative.examples.gemma3 import gemma3
    from litert_torch.generative.layers import kv_cache
    from litert_torch.generative.utilities import converter
    from litert_torch.generative.utilities.export_config import ExportConfig

    builders = {
        "270m": gemma3.build_model_270m,
        "1b": gemma3.build_model_1b,
    }
    if GEMMA3_SIZE not in builders:
        raise ValueError(f"Unsupported GEMMA3_SIZE={GEMMA3_SIZE!r}. Use '270m' or '1b'.")

    print("Building PyTorch model from:", HF_MODEL_DIR)
    pytorch_model = builders[GEMMA3_SIZE](str(HF_MODEL_DIR))

    export_config = ExportConfig()
    export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
    export_config.mask_as_input = True

    print("Converting to LiteRT...")
    converter.convert_to_tflite(
        pytorch_model,
        output_path=str(LITERT_DIR),
        output_name_prefix=ARTIFACT_NAME,
        prefill_seq_len=PREFILL_SEQ_LEN,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
        quantize=QUANTIZE,
        export_config=export_config,
    )
    print("LiteRT conversion complete.")

In [ ]:
tflite_candidates = sorted(
    LITERT_DIR.rglob("*.tflite"), key=lambda path: path.stat().st_mtime, reverse=True
)
if not tflite_candidates:
    raise FileNotFoundError(f"No .tflite files were found under {LITERT_DIR}.")

TFLITE_MODEL = tflite_candidates[0]
print("Selected LiteRT model:", TFLITE_MODEL)
print("Size MB:", round(TFLITE_MODEL.stat().st_size / (1024 * 1024), 2))

print("All discovered .tflite files:")
for path in tflite_candidates:
    print("-", path.name, round(path.stat().st_size / (1024 * 1024), 2), "MB")

## Bundle LiteRT and tokenizer into a `.task` file

The prompt prefix/suffix matches the Google Gemma 3 conversion guide. If your installed MediaPipe package is too old to accept `prompt_prefix` and `prompt_suffix`, the cell falls back to bundling without them and prints a warning; update MediaPipe if you want MediaPipe-managed prompt formatting.

In [ ]:
from mediapipe.tasks.python.genai import bundler

if TASK_FILE.exists():
    TASK_FILE.unlink()

bundle_kwargs = {
    "tflite_model": str(TFLITE_MODEL),
    "tokenizer_model": str(TOKENIZER_MODEL),
    "start_token": "<bos>",
    "stop_tokens": ["<eos>", "<end_of_turn>"],
    "output_filename": str(TASK_FILE),
    "prompt_prefix": "<start_of_turn>user\n",
    "prompt_suffix": "<end_of_turn>\n<start_of_turn>model\n",
}

try:
    bundle_config = bundler.BundleConfig(**bundle_kwargs)
    BUNDLED_PROMPT_TEMPLATE = True
except TypeError as exc:
    if "prompt_prefix" not in str(exc) and "prompt_suffix" not in str(exc):
        raise
    print("Installed MediaPipe does not support prompt_prefix/prompt_suffix in BundleConfig.")
    print("Bundling without embedded prompt template. Upgrade MediaPipe for MediaPipe-managed templates.")
    bundle_kwargs.pop("prompt_prefix")
    bundle_kwargs.pop("prompt_suffix")
    bundle_config = bundler.BundleConfig(**bundle_kwargs)
    BUNDLED_PROMPT_TEMPLATE = False

bundler.create_bundle(bundle_config)

if not TASK_FILE.exists():
    raise FileNotFoundError(f"Bundle did not produce {TASK_FILE}")

print("Created task bundle:", TASK_FILE)
print("Size MB:", round(TASK_FILE.stat().st_size / (1024 * 1024), 2))
print("Bundled prompt template:", BUNDLED_PROMPT_TEMPLATE)

## Write local conversion metadata

These files are uploaded with the artifact so the Hugging Face repo records exactly how the `.task` was produced.

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


metadata = {
    "source_model_id": SOURCE_MODEL_ID,
    "target_repo_id": TARGET_REPO_ID,
    "artifact_name": ARTIFACT_NAME,
    "format": "MediaPipe .task",
    "flutter_gemma_model_file_type": "ModelFileType.task",
    "flutter_gemma_model_type": "ModelType.functionGemma",
    "gemma3_size": GEMMA3_SIZE,
    "quantize": QUANTIZE,
    "prefill_seq_len": PREFILL_SEQ_LEN,
    "kv_cache_max_len": KV_CACHE_MAX_LEN,
    "bundled_prompt_template": BUNDLED_PROMPT_TEMPLATE,
    "tflite_model": str(TFLITE_MODEL),
    "task_file": str(TASK_FILE),
    "task_size_bytes": TASK_FILE.stat().st_size,
    "task_sha256": sha256_file(TASK_FILE),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "references": [
        "https://ai.google.dev/gemma/docs/conversions/hf-to-mediapipe-task",
        "https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference",
        "https://pub.dev/packages/flutter_gemma",
    ],
}

METADATA_FILE.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

task_url = f"https://huggingface.co/{TARGET_REPO_ID}/resolve/main/{TASK_FILE.name}"
card = f"""# {ARTIFACT_NAME} for Flutter Gemma

MediaPipe `.task` conversion of `{SOURCE_MODEL_ID}` for `flutter_gemma` mobile inference.

## Artifact

- Task bundle: `{TASK_FILE.name}`
- Flutter Gemma file type: `ModelFileType.task`
- Flutter Gemma model type: `ModelType.functionGemma`
- Quantization: `{QUANTIZE}`
- Prefill sequence length: `{PREFILL_SEQ_LEN}`
- KV cache max length: `{KV_CACHE_MAX_LEN}`
- Prompt template embedded: `{BUNDLED_PROMPT_TEMPLATE}`
- SHA-256: `{metadata['task_sha256']}`

Download URL:

```text
{task_url}
```

## Flutter Gemma usage

```dart
await FlutterGemma.installModel(modelType: ModelType.functionGemma)
  .fromNetwork('{task_url}')
  .install();
```

Use the banking tool schema and `systemInstruction` from this project when creating the chat.

## Conversion references

- Google Gemma HF SafeTensors to MediaPipe Task guide: https://ai.google.dev/gemma/docs/conversions/hf-to-mediapipe-task
- Google MediaPipe LLM Inference guide: https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference
- Flutter Gemma docs: https://pub.dev/packages/flutter_gemma

## License

This converted artifact follows the license and use restrictions of the source Gemma/FunctionGemma model. Review the source model card and Gemma terms before redistribution or production use.
"""
CARD_FILE.write_text(card, encoding="utf-8")

print("Metadata:", METADATA_FILE)
print("Flutter Gemma artifact note:", CARD_FILE)
print("Expected HF task URL:", task_url)

## Publish the converted model to Hugging Face

Set `HF_TOKEN` with write access and `UPLOAD_TO_HF=1` before running this cell. By default, the `.task` file is uploaded at the repo root so Flutter can use:

`https://huggingface.co/<repo-id>/resolve/main/<artifact-name>.task`

The conversion note is uploaded to `flutter_gemma/README.md` by default to avoid overwriting an existing model card. Set `UPLOAD_CARD_TO_ROOT=1` only when this is a dedicated converted-model repo.

In [ ]:
from huggingface_hub import HfApi, create_repo

upload_to_hf = UPLOAD_TO_HF or os.getenv("UPLOAD_TO_HF", "0") == "1"
hf_upload_token = HF_TOKEN or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
private_repo = PRIVATE_REPO or os.getenv("HF_PRIVATE_REPO", "0") == "1"
upload_raw_tflite = UPLOAD_RAW_TFLITE or os.getenv("UPLOAD_RAW_TFLITE", "1") == "1"
upload_card_to_root = UPLOAD_CARD_TO_ROOT or os.getenv("UPLOAD_CARD_TO_ROOT", "0") == "1"

if not upload_to_hf:
    print("UPLOAD_TO_HF=0; skipping Hugging Face upload.")
    print("To publish, set UPLOAD_TO_HF=1 and rerun this cell.")
elif not hf_upload_token:
    raise RuntimeError("Set HF_TOKEN or HUGGINGFACE_TOKEN with write access before uploading.")
else:
    api = HfApi(token=hf_upload_token)
    create_repo(
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        private=private_repo,
        exist_ok=True,
        token=hf_upload_token,
    )

    api.upload_file(
        path_or_fileobj=str(TASK_FILE),
        path_in_repo=TASK_FILE.name,
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        commit_message="Add Flutter Gemma MediaPipe task bundle",
    )
    api.upload_file(
        path_or_fileobj=str(METADATA_FILE),
        path_in_repo=f"flutter_gemma/{METADATA_FILE.name}",
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        commit_message="Add Flutter Gemma conversion metadata",
    )

    card_path_in_repo = "README.md" if upload_card_to_root else "flutter_gemma/README.md"
    api.upload_file(
        path_or_fileobj=str(CARD_FILE),
        path_in_repo=card_path_in_repo,
        repo_id=TARGET_REPO_ID,
        repo_type="model",
        commit_message="Add Flutter Gemma conversion notes",
    )

    if upload_raw_tflite:
        api.upload_folder(
            folder_path=str(LITERT_DIR),
            path_in_repo="tflite_raw",
            repo_id=TARGET_REPO_ID,
            repo_type="model",
            commit_message="Add raw LiteRT conversion outputs",
        )
        api.upload_file(
            path_or_fileobj=str(TOKENIZER_MODEL),
            path_in_repo="tflite_raw/tokenizer.model",
            repo_id=TARGET_REPO_ID,
            repo_type="model",
            commit_message="Add tokenizer for rebundling",
        )

    print("Published Flutter Gemma task bundle.")
    print(f"https://huggingface.co/{TARGET_REPO_ID}/resolve/main/{TASK_FILE.name}")

## Flutter smoke-test target

After upload, install the exact `.task` URL with `ModelType.functionGemma`, pass this project's banking tools to `createChat(...)`, and smoke test with:

```text
Cho minh xem lich su giao dich 7 ngay gan day
```

Expected first model event: a function call to `get_account_info` with `account_id=ACC_USER`, `info_type=transactions`, and a recent date window based on the session `systemInstruction` date.